# **GPT-1 실습**
: Implementation of GPT-1 model, including pre-training & fine-tuning process. Pre-trained on WikiText2, fine-tuned on IMDB Dataset.
## Improving Language Understanding by Generative Pre-Training

⏩ 논문링크: https://www.mikecaptain.com/resources/pdf/GPT-1.pdf


⏩ 깃허브: https://github.com/tony3ynot/GPT-1/blob/main/GPT_1.ipynb 코드를 참고했습니다


📑 추가로 참고해볼만한 깃허브: https://github.com/lyeoni/gpt-pytorch



In [ ]:
import torch
import torch.nn as nn
from einops import rearrange

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#1. Model Architecture




## 1-1. Transformer Decoder

In [ ]:
### Multi-Head Attention
class MHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads

        # 입력 벡터 변환용 선형 레이어 (Query, Key, Value)
        self.fc_q = nn.Linear(d_model, d_model)
        self.fc_k = nn.Linear(d_model, d_model)
        self.fc_v = nn.Linear(d_model, d_model)

        # 최종 출력 선형 레이어
        self.fc = nn.Linear(d_model, d_model)

        # 스코어 스케일링 상수 (루트 d_k)
        self.scale = torch.sqrt(torch.tensor(d_model/n_heads))

    def forward(self, Q, K, V, mask = None):
        # Q, K, V 벡터 투영
        Q = self.fc_q(Q)
        K = self.fc_k(K)
        V = self.fc_v(V)

        # 헤드 수에 맞춰 차원 재배치 (B: 배치, L: 길이, H: 헤드, D: 헤드당 차원)
        Q = rearrange(Q, 'B L (H D) -> B H L D', H = self.n_heads)
        K = rearrange(K, 'B L (H D) -> B H L D', H = self.n_heads)
        V = rearrange(V, 'B L (H D) -> B H L D', H = self.n_heads)

        # 1. 유사도 스코어 계산 (Dot-product)
        attention_score = Q @ K.transpose(-2, -1)

        # 2. 스케일링
        attention_score = attention_score / self.scale

        # 3. 마스킹 (패딩 및 미래 토큰 마스킹 처리)
        if mask is not None:
            mask = mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)
            attention_score.masked_fill_(mask, -1e9)

        # 4. 소프트맥스를 통한 어텐션 가중치 산출
        attention_weights = torch.softmax(attention_score, dim=-1)

        # 5. 가중치와 Value 행렬곱
        attention = attention_weights @ V

        # 헤드 결합 및 최종 선형 변환
        x = rearrange(attention, 'B H L D -> B L (H D)')
        output = self.fc(x)

        return output


### Feed Forward Network
class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        # 두 개의 선형 레이어와 활성화 함수(GELU)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

        # 가중치 초기화 (Xavier Normal)
        nn.init.xavier_normal_(self.linear1.weight)
        nn.init.xavier_normal_(self.linear2.weight)

    def forward(self, x):
        # d_ff 차원 확장 -> GELU -> 원래 차원으로 복구
        x = self.gelu(self.linear1(x))
        output = self.linear2(x)

        return output


### Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, resid_drop):
        super().__init__()

        # 어텐션 및 피드포워드 계층 구성 요소
        self.mha = MHA(d_model, n_heads)
        self.dropout1 = nn.Dropout(resid_drop)
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-5)

        self.ffn = FFN(d_model, d_ff)
        self.dropout2 = nn.Dropout(resid_drop)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-5)

    def forward(self, x, attn_mask):
        # Multi-Head Attention + 잔차 연결 + 레이어 정규화
        residual = self.mha(x, x, x, attn_mask)
        residual = self.dropout1(residual)
        x = self.layernorm1(x + residual)

        # Feed Forward Network + 잔차 연결 + 레이어 정규화
        residual = self.ffn(x)
        residual = self.dropout2(residual)
        output = self.layernorm2(x + residual)

        return output


### Decoder
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, seq_len, d_model, n_layers, n_heads, d_ff, embd_drop, resid_drop, pad_id):
        super().__init__()

        self.pad_id = pad_id

        # 단어 및 학습 가능한 위치 임베딩 설정
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(embd_drop)
        self.pos_embedding = nn.Embedding(seq_len+1, d_model)

        # n_layers만큼 데코더 레이어 스택 구성
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, resid_drop) for _ in range(n_layers)])

        nn.init.xavier_normal_(self.embedding.weight)

    def forward(self, x):
        # 위치 인덱스 생성 및 패딩 마스킹 처리
        positions = torch.arange(x.size(1), device=x.device).repeat(x.size(0), 1) + 1
        position_pad_mask = x.eq(self.pad_id)
        positions.masked_fill_(position_pad_mask, 0)

        # 단어 임베딩과 위치 임베딩 합산
        output = self.dropout(self.embedding(x)) + self.pos_embedding(positions)

        # 패딩 마스크 및 미래 토큰 마스크 생성 후 병합
        pad_mask = self.get_padding_mask(x, x, self.pad_id)
        future_mask = self.get_future_mask(x).to(device=pad_mask.device)
        attn_mask = torch.gt((pad_mask.to(dtype=future_mask.dtype) + future_mask), 0)

        # 모든 레이어 순차 통과
        for layer in self.layers:
            output = layer(output, attn_mask)

        return output

    # 패딩 위치 마스크 생성 함수
    def get_padding_mask(self, q, k, pad_id):
        pad_mask = k.eq(pad_id).unsqueeze(1).repeat(1, q.size(1), 1)
        return pad_mask

    # 미래 정보 차단을 위한 삼각 행렬 마스크 생성 함수
    def get_future_mask(self, q):
        bs, q_len = q.size()
        future_mask = torch.ones(bs, q_len, q_len).triu(diagonal=1)
        return future_mask

## 1-2. GPT-1

In [ ]:
### GPT-1
# GPT-1 메인 모델 구조
class GPT(nn.Module):
    def __init__(self,
                 vocab_size, # 단어 사전 크기
                 seq_len = 512, # 최대 입력 토큰 수
                 d_model = 768, # 데이터 표현 차원
                 n_layers = 12, # 디코더 블록 층수
                 n_heads = 12, # 멀티 헤드 수
                 d_ff = 3072, # 피드 포워드 은닉층 크기
                 embd_drop = 0.1,
                 resid_drop = 0.1,
                 pad_id = 0):
        super().__init__()

        # TransformerDecoder 인스턴스 생성 및 설정 전달
        self.decoder = TransformerDecoder(vocab_size, seq_len, d_model, n_layers, n_heads,
                                          d_ff, embd_drop, resid_drop, pad_id)

    def forward(self, x):
        # 디코더를 통한 특징 추출
        outputs = self.decoder(x)

        return outputs


### Language Model (pre-training)
class GPTLMHead(nn.Module):
    def __init__(self, gpt):
        super().__init__()
        # GPT 모델의 임베딩 가중치 크기 참조
        vocab_size, d_model = gpt.decoder.embedding.weight.size()

        self.gpt = gpt
        # 다음 단어 예측을 위한 선형 레이어 (Weight Tying 적용)
        self.linear = nn.Linear(d_model, vocab_size, bias = False)
        self.linear.weight = gpt.decoder.embedding.weight

    def forward(self, x):
        # GPT를 통한 시퀀스 특징 추출
        x = self.gpt(x)

        # 각 시점별 단어 예측 로짓 계산
        lm_logits = self.linear(x)

        return lm_logits


### Classification Model (fine-tuning)
class GPTClsHead(nn.Module):
    def __init__(self, gpt, n_class, cls_token_id, cls_drop=0.1):
        super().__init__()
        vocab_size, d_model = gpt.decoder.embedding.weight.size()
        self.cls_token_id = cls_token_id

        self.gpt = gpt

        # LM Head: 사전 학습 시 사용한 언어 모델 출력층
        self.linear1 = nn.Linear(d_model, vocab_size, bias=False)
        self.linear1.weight = gpt.decoder.embedding.weight

        # Cls Head: 분류 작업을 위한 최종 출력층
        self.linear2 = nn.Linear(d_model, n_class)
        self.dropout = nn.Dropout(cls_drop)

        # 분류 레이어 가중치 초기화
        nn.init.normal_(self.linear2.weight, std=0.02)
        nn.init.normal_(self.linear2.bias, 0)

    def forward(self, x):
        # GPT 특징 추출
        outputs = self.gpt(x)

        # 보조 목적 함수를 위한 언어 모델 로짓 산출
        lm_logits = self.linear1(outputs)

        # 특수 토큰(<cls>) 위치의 출력값만 추출하여 분류 수행
        outputs = outputs[x.eq(self.cls_token_id)]
        cls_logits = self.linear2(self.dropout(outputs))

        return lm_logits, cls_logits

# 2. Training

## 2-1. Pre-training

In [ ]:
# 필수 라이브러리(transformers, datasets, tokenizers) 설치
!pip install transformers datasets tokenizers

# PyTorch 및 데이터 처리 관련 모듈 임포트
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import numpy as np
from tqdm import tqdm

# 연산 수행 장치 설정 (GPU 사용 가능 시 cuda, 불가 시 cpu)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
### WikiText Dataset class
class WikiTextDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        # 데이터셋, 토크나이저, 시퀀스 길이 초기화
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        # 전체 데이터 개수 반환
        return len(self.data)

    def __getitem__(self, idx):
        # 인덱스에 해당하는 텍스트 추출 및 인코딩
        text = self.data[idx]['text']
        encoded = self.tokenizer.encode(text)
        input_ids = encoded.ids

        # 설정된 시퀀스 길이에 맞춰 자르거나 패딩(0) 추가
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))

        # 다음 단어 예측(Next-word prediction)을 위한 입력 및 타겟 생성
        # inputs: 마지막 토큰 제외, targets: 첫 번째 토큰 제외
        inputs = torch.tensor(input_ids[:-1])
        targets = torch.tensor(input_ids[1:])

        return inputs, targets

In [ ]:
### Hyper-parameters
# 학습 및 모델 설정을 위한 주요 하이퍼파라미터 정의
VOCAB_SIZE = 10000     # 단어 사전 크기
SEQ_LEN = 512          # 최대 문장 길이
BATCH_SIZE = 8         # 배치 크기
EPOCHS = 3             # 전체 데이터 반복 학습 횟수
LEARNING_RATE = 1e-4   # 옵티마이저 학습률

### Tokenizer Training
# WikiText 데이터셋 로드 및 BPE 기반 토크나이저 초기화
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
tokenizer = Tokenizer(BPE())
# 패딩 및 분류 특수 토큰을 포함한 학습기 설정
trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<pad>", "<cls>"])
tokenizer.pre_tokenizer = Whitespace()

# 학습을 위한 텍스트 코퍼스 생성기 정의
def get_training_corpus():
    for i in range(0, len(dataset['train'])):
        yield dataset['train'][i]['text']

# 이터레이터를 활용한 토크나이저 학습 수행
tokenizer.train_from_iterator(get_training_corpus(), trainer)

### Dataset Setup
# 학습 데이터셋 객체 생성 및 데이터 로더 설정 (입력/타겟 분리를 위해 SEQ_LEN + 1 적용)
train_dataset = WikiTextDataset(dataset['train'], tokenizer, SEQ_LEN + 1)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
### Model Initialization
# 설정된 보컬 사이즈 및 시퀀스 길이를 바탕으로 GPT 언어 모델 초기화 및 장치 할당
model = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)

# 가중치 감쇠(Weight Decay)가 포함된 AdamW 옵티마이저 설정
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# 학습 진행에 따라 학습률을 코사인 그래프 형태로 조절하는 스케줄러 설정
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_dataloader) * EPOCHS)

In [ ]:
### Pre-Training
for epoch in range(EPOCHS):
    # 모델을 학습 모드로 설정
    model.train()
    total_loss = 0

    # 학습 진행 상황 시각화를 위한 프로그레스 바 설정
    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        # 데이터를 연산 장치(GPU/CPU)로 이동
        inputs, targets = inputs.to(device), targets.to(device)

        # 순전파 연산 및 로짓 계산
        logits = model(inputs)
        # 패딩 토큰(ignore_index=0)을 제외한 교차 엔트로피 손실 계산
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=0)

        # 기울기 초기화, 역전파 수행, 가중치 업데이트
        optimizer.zero_grad()
        loss.backward()
        # 안정적인 학습을 위한 그래디언트 클리핑 적용
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        # 학습률 스케줄러 업데이트
        scheduler.step()

        # 실시간 손실 기록 및 프로그레스 바 갱신
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})

    # 에폭별 평균 손실 출력
    avg_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("Training completed!")

# 학습된 모델 상태 및 학습 정보 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'final_loss': avg_loss
}, 'gpt1_pretrained.pt')

Epoch 1/3: 100%|██████████| 4590/4590 [1:01:14<00:00,  1.25it/s, loss=7.17]



Epoch 1 Average Loss: 7.1747


Epoch 2/3: 100%|██████████| 4590/4590 [1:01:18<00:00,  1.25it/s, loss=7.11]



Epoch 2 Average Loss: 7.1083


Epoch 3/3:  68%|██████▊   | 3107/4590 [41:27<19:42,  1.25it/s, loss=nan]

## 2-2. Fine-tuning

In [ ]:
### IMDB Dataset class
class IMDBDataset(Dataset):
    def __init__(self, data, tokenizer, seq_len):
        # 데이터, 토크나이저, 시퀀스 길이 초기화
        self.data = data
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        # 전체 데이터 개수 반환
        return len(self.data)

    def __getitem__(self, idx):
        # 텍스트와 정답 레이블(긍정/부정) 추출
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        # 문장 시작 지점에 분류용 <cls> 토큰 추가 및 인코딩
        encoded = self.tokenizer.encode("<cls> " + text)
        input_ids = encoded.ids

        # 설정된 길이에 맞춰 데이터 자르기 또는 패딩(0) 추가
        if len(input_ids) > self.seq_len:
            input_ids = input_ids[:self.seq_len]
        else:
            input_ids = input_ids + [0] * (self.seq_len - len(input_ids))

        # 텐서 형태로 변환하여 반환
        return torch.tensor(input_ids), torch.tensor(label)

In [ ]:
### Dataset Setup
# IMDB 데이터셋 로드
from datasets import load_dataset
imdb_dataset = load_dataset('imdb')

# 학습 및 검증 데이터셋 객체 생성 및 데이터 로더 설정
train_dataset = IMDBDataset(imdb_dataset['train'], tokenizer, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataset = IMDBDataset(imdb_dataset['test'], tokenizer, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=4)

In [ ]:
# 사전 학습된 가중치 로드를 위한 기본 모델 초기화 및 할당
premodel = GPTLMHead(GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)).to(device)
# 저장된 사전 학습 가중치 파일('gpt1_pretrained.pt') 로드
premodel.load_state_dict(torch.load('gpt1_pretrained.pt'))

### Fine-tuning Model Initialization
# 분류 작업을 위한 모델 초기화 (사전 학습된 GPT 엔진 활용)
model = GPTClsHead(
    gpt=premodel.gpt,  # 사전 학습된 GPT 모델 추출
    n_class=2,         # 분류할 클래스 수 (긍정/부정)
    cls_token_id=tokenizer.token_to_id("<cls>"), # <cls> 토큰의 고유 ID 설정
    cls_drop=0.1       # 분류층 드롭아웃 확률
).to(device)

# 파인튜닝을 위한 옵티마이저 설정 (사전 학습보다 낮은 학습률 사용)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
### Fine-tuning
# 파인튜닝 하이퍼파라미터 및 최고 정확도 기록용 변수 설정
EPOCHS = 3
auxiliary_ratio = 0.5  # 보조 언어 모델 손실 비중
best_acc = 0

for epoch in range(EPOCHS):
    ## Training
    # 모델을 학습 모드로 설정 및 손실 누적 변수 초기화
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        # 데이터 장치 할당
        inputs, labels = inputs.to(device), labels.to(device)

        # 모델 출력 (언어 모델 로짓과 분류 로짓 동시 산출)
        lm_logits, cls_logits = model(inputs)
        # 타겟 시퀀스 길이에 맞춰 언어 모델 로짓 슬라이싱
        lm_logits = lm_logits[:, :-1].contiguous()

        ## Loss Function w/ Auxiliary Function
        # 보조 언어 모델 손실(Auxiliary Loss) 계산
        lm_loss = F.cross_entropy(lm_logits.view(-1, lm_logits.size(-1)),
                                  inputs[:, 1:].contiguous().view(-1), ignore_index=0)
        # 분류 작업 손실(Classification Loss) 계산
        cls_loss = F.cross_entropy(cls_logits, labels)
        # 최종 통합 손실 계산 (분류 손실 + 가중치 적용된 보조 손실)
        loss = cls_loss + (auxiliary_ratio * lm_loss)

        # 역전파 및 가중치 업데이트
        optimizer.zero_grad()
        loss.backward()
        # 그래디언트 클리핑으로 안정성 확보
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        # 손실 기록 및 프로그레스 바 상태 업데이트
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': total_loss / (batch_idx + 1)})

    ## Validation
    # 검증 수행을 위해 모델을 평가 모드로 전환
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            # 분류 로짓만 획득
            _, cls_logits = model(inputs)

            # 예측값과 실제 정답 비교하여 맞은 개수 카운트
            predictions = torch.argmax(cls_logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    # 에폭 종료 후 검증 정확도 계산 및 출력
    accuracy = correct / total
    print(f"Epoch {epoch+1} Validation Accuracy: {accuracy:.4f}")

    # 검증 정확도가 이전 최고치보다 높을 경우 모델 가중치 저장
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), 'gpt1_imdb_best.pt')

print(f"Fine-tuning completed! Best accuracy: {best_acc:.4f}")

# Test

In [ ]:
# 분류 작업을 위한 모델 인스턴스 생성 (GPT 엔진 및 분류 헤드 설정)
model = GPTClsHead(
    GPT(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN), # 기본 GPT 모델 구조
    n_class=2,                                  # 분류 클래스 개수 (긍정/부정)
    cls_token_id=tokenizer.token_to_id("<cls>"), # 분류용 특수 토큰 ID 추출
    cls_drop=0.1                                # 분류층 드롭아웃 비율
)

# 학습이 완료된 최적의 모델 가중치 파일 로드
model.load_state_dict(torch.load('gpt1_imdb_best.pt'))

In [ ]:
# Test
def predict_sentiment(text):
    # 모델을 평가 모드로 전환
    model.eval()

    # 문장 시작에 <cls> 토큰 추가 후 인코딩
    encoded = tokenizer.encode("<cls> " + text)
    input_ids = encoded.ids

    # 시퀀스 길이를 SEQ_LEN에 맞춰 조절 (자르기 또는 패딩 추가)
    if len(input_ids) > SEQ_LEN:
        input_ids = input_ids[:SEQ_LEN]
    else:
        input_ids = input_ids + [0] * (SEQ_LEN - len(input_ids))

    # 입력을 텐서로 변환하여 장치에 할당
    inputs = torch.tensor([input_ids]).to(device)

    # 기울기 계산 없이 추론 수행
    with torch.no_grad():
        _, cls_logits = model(inputs)
        # 가장 높은 확률을 가진 클래스 선택
        prediction = torch.argmax(cls_logits, dim=-1)

    # 인덱스 값에 따라 긍정(1) 또는 부정(0) 결과 반환
    return "Positive" if prediction.item() == 1 else "Negative"

# Example
# 테스트용 예시 문장 정의 및 결과 출력
test_text = "This movie was really great! I enjoyed every moment of it."
print(f"Text: {test_text}")
print(f"Sentiment: {predict_sentiment(test_text)}")